In [4]:
%pip install XGBoost

Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
import numpy as np
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [8]:
df = pd.read_csv('../datasets/creditcard.csv')
df.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,400000,1,1,2,32,0,0,0,0,0,0,55773,55917,51389,48272,49478,51242,3028,3023,3000,3000,3000,38662,0
1,120000,2,2,2,30,-1,-1,-1,-1,-1,-1,140,3230,3011,1964,1883,1538,3230,3011,1964,1883,1538,1911,0
2,270000,2,2,2,32,0,0,0,0,0,0,59710,49986,104390,94856,86461,83650,1808,69563,2891,2689,3012,2771,0
3,280000,2,2,1,27,0,0,0,0,0,0,280913,283222,273160,257689,193231,191143,11052,9563,15017,5374,5420,6021,0
4,30000,2,1,2,27,0,0,-1,0,0,-2,1512,2458,664,1814,0,0,1000,664,1500,0,0,0,0


In [11]:
report = sv.analyze(df)
report.show_html('df_report.html')

                                             |          | [  0%]   00:00 -> (? left)

Report df_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


EDA : It is not clear what some of the variables mean in terms of the dataset. There are no missing values which is nice. The data is very imbalanced.

In [13]:
# Data preparation
X = df.drop('default payment next month', axis=1)
y = df['default payment next month']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [14]:
# RandomForest Model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predictions
rf_pred = rf_model.predict(X_test)

# Evaluation
print("RandomForest Results:")
print("Accuracy:", accuracy_score(y_test, rf_pred))
print("Classification Report:")
print(classification_report(y_test, rf_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

RandomForest Results:
Accuracy: 0.8114583333333333
Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      3738
           1       0.62      0.38      0.47      1062

    accuracy                           0.81      4800
   macro avg       0.73      0.66      0.68      4800
weighted avg       0.79      0.81      0.79      4800

Confusion Matrix:
[[3496  242]
 [ 663  399]]


In [15]:
# XGBoost Model
xgb_model = XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

# Predictions
xgb_pred = xgb_model.predict(X_test)

# Evaluation
print("XGBoost Results:")
print("Accuracy:", accuracy_score(y_test, xgb_pred))
print("Classification Report:")
print(classification_report(y_test, xgb_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, xgb_pred))

XGBoost Results:
Accuracy: 0.8102083333333333
Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.93      0.88      3738
           1       0.61      0.38      0.47      1062

    accuracy                           0.81      4800
   macro avg       0.73      0.66      0.68      4800
weighted avg       0.79      0.81      0.79      4800

Confusion Matrix:
[[3483  255]
 [ 656  406]]


c:\Users\tyler\anaconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [13:28:39] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [ ]:
# Base XGBoost evaluation on the test set
xgb_base = XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb_base.fit(X_train, y_train)
xgb_base_test_pred = xgb_base.predict(X_test)

print("Base XGBoost Test Set Results:")
print("Accuracy:", accuracy_score(y_test, xgb_base_test_pred))
print("F1 Score:", classification_report(y_test, xgb_base_test_pred, output_dict=True)['weighted avg']['f1-score'])
print("Classification Report:")
print(classification_report(y_test, xgb_base_test_pred))

# 5-fold cross-validation across the full dataset
xgb_cv = XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb_cv_scores = cross_val_score(xgb_cv, X, y, cv=5, scoring='f1')
print("\n5-fold CV F1 scores:", xgb_cv_scores)
print(f"Mean F1: {xgb_cv_scores.mean():.3f}")
print(f"Std F1: {xgb_cv_scores.std():.3f}")

# XGBoost performance summary
The base XGBoost model performed well on the test set, and the 5-fold cross-validation gives a more stable estimate of how it generalizes.

- The mean F1 score across folds represents the average performance we can expect on unseen data.
- The standard deviation shows how much the model’s performance varies from fold to fold; a lower value means the results are more consistent and reliable.

Together, mean and standard deviation help indicate whether the model is robust or if its performance is sensitive to the specific training split used.

In [24]:

# RandomForest with class_weight='balanced'
rf_balanced = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_cv_scores = cross_val_score(rf_balanced, X_train, y_train, cv=5, scoring='f1')
print(f"RandomForest CV F1 scores: {rf_cv_scores}")
print(f"RandomForest CV F1 mean: {rf_cv_scores.mean():.3f} (+/- {rf_cv_scores.std() * 2:.3f})")

# XGBoost with scale_pos_weight
xgb_balanced = XGBClassifier(n_estimators=100, random_state=42, class_weight='balanced', eval_metric='logloss')
xgb_cv_scores = cross_val_score(xgb_balanced, X_train, y_train, cv=5, scoring='f1')
print(f"XGBoost CV F1 scores: {xgb_cv_scores}")
print(f"XGBoost CV F1 mean: {xgb_cv_scores.mean():.3f} (+/- {xgb_cv_scores.std() * 2:.3f})")

RandomForest CV F1 scores: [0.42424242 0.44359756 0.42286609 0.45714286 0.43310131]
RandomForest CV F1 mean: 0.436 (+/- 0.026)


c:\Users\tyler\anaconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [13:51:27] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "class_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\tyler\anaconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [13:51:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "class_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\tyler\anaconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [13:51:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "class_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\tyler\anaconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [13:51:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "class_weight" } are not used.

  bst.update(dtrain, iteration=i, fo

XGBoost CV F1 scores: [0.43366187 0.4729927  0.43674699 0.47410649 0.46877296]
XGBoost CV F1 mean: 0.457 (+/- 0.036)


c:\Users\tyler\anaconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [13:51:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "class_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


I do not think my AI did this correctly. This drastically worsened the model when I think it should do the opposite or have little change.

In [25]:
# Train balanced models on full training data and evaluate on test
print("\n--- Balanced Models Evaluation on Test Set ---")

# RandomForest balanced
rf_balanced.fit(X_train, y_train)
rf_balanced_pred = rf_balanced.predict(X_test)
print("RandomForest Balanced Results:")
print("Accuracy:", accuracy_score(y_test, rf_balanced_pred))
print("F1 Score:", classification_report(y_test, rf_balanced_pred, output_dict=True)['weighted avg']['f1-score'])
print("Classification Report:")
print(classification_report(y_test, rf_balanced_pred))

# XGBoost balanced
xgb_balanced.fit(X_train, y_train)
xgb_balanced_pred = xgb_balanced.predict(X_test)
print("\nXGBoost Balanced Results:")
print("Accuracy:", accuracy_score(y_test, xgb_balanced_pred))
print("F1 Score:", classification_report(y_test, xgb_balanced_pred, output_dict=True)['weighted avg']['f1-score'])
print("Classification Report:")
print(classification_report(y_test, xgb_balanced_pred))

# Comparison
print("\n--- Performance Impact ---")
print("RandomForest:")
print(f"  Original F1: {classification_report(y_test, rf_pred, output_dict=True)['weighted avg']['f1-score']:.3f}")
print(f"  Balanced F1: {classification_report(y_test, rf_balanced_pred, output_dict=True)['weighted avg']['f1-score']:.3f}")
print("XGBoost:")
print(f"  Original F1: {classification_report(y_test, xgb_pred, output_dict=True)['weighted avg']['f1-score']:.3f}")
print(f"  Balanced F1: {classification_report(y_test, xgb_balanced_pred, output_dict=True)['weighted avg']['f1-score']:.3f}")


--- Balanced Models Evaluation on Test Set ---
RandomForest Balanced Results:
Accuracy: 0.8154166666666667
F1 Score: 0.7944064608347629
Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      3738
           1       0.65      0.36      0.46      1062

    accuracy                           0.82      4800
   macro avg       0.74      0.65      0.68      4800
weighted avg       0.80      0.82      0.79      4800


XGBoost Balanced Results:
Accuracy: 0.8102083333333333


c:\Users\tyler\anaconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [13:51:42] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "class_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


F1 Score: 0.7929538132021267
Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.93      0.88      3738
           1       0.61      0.38      0.47      1062

    accuracy                           0.81      4800
   macro avg       0.73      0.66      0.68      4800
weighted avg       0.79      0.81      0.79      4800


--- Performance Impact ---
RandomForest:
  Original F1: 0.793
  Balanced F1: 0.794
XGBoost:
  Original F1: 0.793
  Balanced F1: 0.793


In [26]:
# XGBoost with bootstrapping (subsample=0.8)
xgb_subsample = XGBClassifier(n_estimators=100, random_state=42, subsample=0.8, eval_metric='logloss')
xgb_subsample.fit(X_train, y_train)
xgb_subsample_pred = xgb_subsample.predict(X_test)

print("XGBoost with subsample=0.8 Results:")
print("Accuracy:", accuracy_score(y_test, xgb_subsample_pred))
print("F1 Score:", classification_report(y_test, xgb_subsample_pred, output_dict=True)['weighted avg']['f1-score'])
print("Classification Report:")
print(classification_report(y_test, xgb_subsample_pred))

# Comparison with original XGBoost
print("\n--- Comparison ---")
print("Original XGBoost:")
print(f"  Accuracy: {accuracy_score(y_test, xgb_pred):.3f}")
print(f"  F1: {classification_report(y_test, xgb_pred, output_dict=True)['weighted avg']['f1-score']:.3f}")
print("XGBoost with subsample=0.8:")
print(f"  Accuracy: {accuracy_score(y_test, xgb_subsample_pred):.3f}")
print(f"  F1: {classification_report(y_test, xgb_subsample_pred, output_dict=True)['weighted avg']['f1-score']:.3f}")

XGBoost with subsample=0.8 Results:
Accuracy: 0.8047916666666667
F1 Score: 0.78796318110991
Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.93      0.88      3738
           1       0.59      0.38      0.46      1062

    accuracy                           0.80      4800
   macro avg       0.72      0.65      0.67      4800
weighted avg       0.78      0.80      0.79      4800


--- Comparison ---
Original XGBoost:
  Accuracy: 0.810
  F1: 0.793
XGBoost with subsample=0.8:
  Accuracy: 0.805
  F1: 0.788


In [27]:
# Manual Hyperparameter Tuning

# Tuned RandomForest (with class_weight='balanced')
rf_tuned = RandomForestClassifier(n_estimators=150, max_depth=20, random_state=42, class_weight='balanced')
rf_tuned.fit(X_train, y_train)
rf_tuned_pred = rf_tuned.predict(X_test)
print("Tuned RandomForest F1:", classification_report(y_test, rf_tuned_pred, output_dict=True)['weighted avg']['f1-score'])

# Tuned XGBoost (with subsample=0.8)
xgb_tuned = XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.1, random_state=42, subsample=0.8, eval_metric='logloss')
xgb_tuned.fit(X_train, y_train)
xgb_tuned_pred = xgb_tuned.predict(X_test)
print("Tuned XGBoost F1:", classification_report(y_test, xgb_tuned_pred, output_dict=True)['weighted avg']['f1-score'])

# Comparison
print("\n--- Tuning Impact ---")
print("RandomForest:")
print(f"  Before: {classification_report(y_test, rf_balanced_pred, output_dict=True)['weighted avg']['f1-score']:.3f}")
print(f"  After: {classification_report(y_test, rf_tuned_pred, output_dict=True)['weighted avg']['f1-score']:.3f}")
print("XGBoost:")
print(f"  Before: {classification_report(y_test, xgb_subsample_pred, output_dict=True)['weighted avg']['f1-score']:.3f}")
print(f"  After: {classification_report(y_test, xgb_tuned_pred, output_dict=True)['weighted avg']['f1-score']:.3f}")

Tuned RandomForest F1: 0.7963710073360879
Tuned XGBoost F1: 0.799353683521334

--- Tuning Impact ---
RandomForest:
  Before: 0.794
  After: 0.796
XGBoost:
  Before: 0.788
  After: 0.799


In [28]:
# Manual Hyperparameter Tuning

# Tuned RandomForest (with class_weight='balanced', increased n_estimators and limited max_depth)
rf_tuned = RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42, class_weight='balanced')
rf_tuned.fit(X_train, y_train)
rf_tuned_pred = rf_tuned.predict(X_test)
print("Tuned RandomForest F1:", classification_report(y_test, rf_tuned_pred, output_dict=True)['weighted avg']['f1-score'])

# Tuned XGBoost (with subsample=0.8, increased n_estimators, adjusted max_depth and learning_rate)
xgb_tuned = XGBClassifier(n_estimators=150, max_depth=4, learning_rate=0.1, random_state=42, subsample=0.8, eval_metric='logloss')
xgb_tuned.fit(X_train, y_train)
xgb_tuned_pred = xgb_tuned.predict(X_test)
print("Tuned XGBoost F1:", classification_report(y_test, xgb_tuned_pred, output_dict=True)['weighted avg']['f1-score'])

# Comparison
print("\n--- Tuning Impact ---")
print("RandomForest:")
print(f"  Before: {classification_report(y_test, rf_balanced_pred, output_dict=True)['weighted avg']['f1-score']:.3f}")
print(f"  After: {classification_report(y_test, rf_tuned_pred, output_dict=True)['weighted avg']['f1-score']:.3f}")
print("XGBoost:")
print(f"  Before: {classification_report(y_test, xgb_subsample_pred, output_dict=True)['weighted avg']['f1-score']:.3f}")
print(f"  After: {classification_report(y_test, xgb_tuned_pred, output_dict=True)['weighted avg']['f1-score']:.3f}")

Tuned RandomForest F1: 0.7870004844587607
Tuned XGBoost F1: 0.7990163592692365

--- Tuning Impact ---
RandomForest:
  Before: 0.794
  After: 0.787
XGBoost:
  Before: 0.788
  After: 0.799


# Tuning summary
The RandomForest tuning decreased performance slightly, which suggests that the chosen depth and estimator settings were too conservative for this dataset. The XGBoost tuning improved performance, showing the chosen parameters helped it generalize better.

- RandomForest: performance decreased after tuning, so these specific parameters were not the best choice.
- XGBoost: performance increased after tuning, so the chosen parameters were more effective.

Overall, I chose reasonable parameters to tune, but RandomForest would benefit from further exploration of `max_depth` and `n_estimators`, while XGBoost tuning appears more promising for this dataset.

## Out of the box XGBoost is better! 

When we started to tune XGBoost too much it can worsen the model quickly however with base models, it outperforms the Random Forest. 